# 02 - Fraud EDA

How do fraudulent transactions **differ** from legitimate ones?

This notebook slices the same cleaned sample by the `isFraud` label and looks
at four angles: amount, transaction type, timing and account balances.

**Prerequisite:** `python main.py` (see `01_data_exploration.ipynb`).

In [ ]:
import sys
from pathlib import Path

# Make 'import src...' work no matter where the kernel was started from.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_transactions.csv"
FEATURED_PATH = PROJECT_ROOT / "data" / "processed" / "featured_transactions.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "fraud_model.pkl"

print("Project root:", PROJECT_ROOT)

In [ ]:
SAMPLE_ROWS = 200_000    # rows to keep for this notebook
RANDOM_SAMPLE = False    # True = uniform random sample (streams the whole file, slower)

if RANDOM_SAMPLE:
    # Streams the full cleaned CSV one chunk at a time and keeps a random
    # slice: memory-bounded and unbiased, but slower than the head sample.
    from src.detection.fraud_model import sample_processed_data
    df = sample_processed_data(max_rows=SAMPLE_ROWS, random_state=42)
else:
    # The cleaned file is ordered by simulation time, so the head covers the
    # first hours of the simulated month. Fast; plenty for structural EDA.
    df = pd.read_csv(DATA_PATH, nrows=SAMPLE_ROWS)

print(df.shape, "| fraud rows in sample:", int(df["isfraud"].sum()))

## 1. Amount: fraud vs legitimate

In [ ]:
df.groupby("isfraud")["amount"].describe().T

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for label, group in df.groupby("isfraud"):
    ax.hist(np.log1p(group["amount"]), bins=60, alpha=0.6,
            label="fraud" if label else "legitimate")
ax.set_xlabel("log1p(amount)")
ax.set_ylabel("transactions")
ax.set_title("Amount distribution by class")
ax.legend()
plt.show()

In [ ]:
# Does a fraudulent transaction drain the origin account?
fraud = df[df["isfraud"] == 1].copy()
drain = np.where(
    fraud["oldbalanceorg"] > 0,
    (fraud["oldbalanceorg"] - fraud["newbalanceorig"]) / fraud["oldbalanceorg"],
    0.0,
)
fraud = fraud.assign(drain_ratio=drain)
fraud["drain_ratio"].describe(percentiles=[0.5, 0.9]).to_frame("fraud: origin balance drained")

## 2. Transaction type

In [ ]:
fraud["type"].value_counts().to_frame("fraud transactions")

In [ ]:
comp = pd.crosstab(df["type"], df["isfraud"], normalize="index") * 100
ax = comp.plot(kind="bar", stacked=True, figsize=(10, 4), rot=0)
ax.set_ylabel("% of transactions")
ax.set_title("Fraud share per transaction type")
plt.show()

## 3. Timing

In [ ]:
by_hour = df.assign(hour=(df["step"] - 1) % 24).groupby("hour")["isfraud"].agg(
    transactions="count", fraud="sum"
)
by_hour["fraud_rate_%"] = 100 * by_hour["fraud"] / by_hour["transactions"]

ax = by_hour["fraud_rate_%"].plot(figsize=(10, 3), title="Fraud rate by simulated hour")
ax.set_xlabel("hour of day")
ax.set_ylabel("fraud rate %")
plt.show()

by_day = df.assign(day=(df["step"] - 1) // 24).groupby("day")["isfraud"].sum()
ax = by_day.plot(figsize=(10, 3), title="Fraudulent transactions per simulated day")
ax.set_xlabel("simulated day")
ax.set_ylabel("fraud count")
plt.show()

## 4. Balance patterns of fraud

Questions worth answering before engineering features:

1. Does the origin account end up emptied?
2. Is the amount comparable to the whole origin balance?
3. Does the destination balance change?

In [ ]:
checks = pd.DataFrame({
    "origin emptied (new balance < 1)": fraud["newbalanceorig"] < 1,
    "amount >= 90% of origin balance": fraud["amount"] >= 0.9 * fraud["oldbalanceorg"],
    "destination balance unchanged": fraud["newbalancedest"] == fraud["oldbalancedest"],
    "origin balance was positive": fraud["oldbalanceorg"] > 0,
})
(checks.mean() * 100).round(1).to_frame("% of fraud rows")

## 5. Existing flags and simple correlations

In [ ]:
print(
    "isFlaggedFraud set on",
    int(df["isflaggedfraud"].sum()),
    "rows of", len(df),
    f"({100 * df['isflaggedfraud'].mean():.4f}%)",
)
print("fraud rows:", int(df["isfraud"].sum()))

In [ ]:
corr = df.select_dtypes("number").corr()["isfraud"].drop("isfraud").sort_values()
ax = corr.plot(kind="barh", figsize=(8, 6), title="Pearson correlation with isFraud")
ax.set_xlabel("correlation")
plt.tight_layout()
plt.show()

## Summary

- **Fraud occurs only for `TRANSFER` and `CASH_OUT`** (money leaving an
  account) - visible in the type table above; `PAYMENT`/`CASH_IN`/`DEBIT` never
  appear. Any model can therefore lean on transaction type, which is why the
  rule engine includes a "sensitive transaction type" rule.
- **Fraud drains the origin balance** - the drain-ratio statistics show most
  fraudulent rows empty (or nearly empty) the customer's account. This motivates
  the `origin_balance_error` / balance-depletion features and rules in `03`.
- **Timing is almost uninformative**: the fraud rate is roughly flat across
  hours and days (PaySim spreads fraud uniformly), so time features help mainly
  with *velocity* (rapid bursts) rather than "fraud happens at night".
- **`isFlaggedFraud` is nearly always 0** - the simulator's own flag catches
  almost nothing and is a useless feature; the models here must do better.
- Amount distributions overlap heavily: no single threshold separates the
  classes, which is exactly why we combine several detectors.